data load and preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn

In [2]:
## giving column names
index_names = ['engine_id', 'cycle']
setting_names = ['setting1', 'setting2', 'setting3']
sensor_names = ['s{}'.format(i) for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

In [3]:
## loading data
train_df = pd.read_csv('data/train_FD001.txt', sep='\s+', header=None, names=col_names)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\USER\AppData\Local\Temp\ipykernel_46952\1899309766.py:2: SyntaxWarning: invalid escape sequence '\s'
  train_df = pd.read_csv('data/train_FD001.txt', sep='\s+', header=None, names=col_names)


In [6]:
# calculating Remaining Useful Life (RUL) 
max_cycle = train_df.groupby('engine_id')['cycle'].max().reset_index()
max_cycle.columns = ['engine_id', 'max_cycle']
train_df = train_df.merge(max_cycle, on='engine_id', how='left')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']
train_df.drop('max_cycle', axis=1, inplace=True)

print("data loading completed , total lines:", len(train_df))

data loading completed , total lines: 20631


### data normalization and sequence making for lstm

In [9]:
# scaling sensor features
scaler = MinMaxScaler()
train_df[sensor_names] = scaler.fit_transform(train_df[sensor_names])

# changing to lstm required format
def create_sequences(data, seq_length, feature_cols):
    X, y = [], []
    for engine_id in data['engine_id'].unique():
        engine_data = data[data['engine_id'] == engine_id]
        if len(engine_data) >= seq_length:
            features = engine_data[feature_cols].values
            target = engine_data['RUL'].values
            for i in range(len(engine_data) - seq_length):
                X.append(features[i:i+seq_length])
                y.append(target[i+seq_length])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df, seq_length=30, feature_cols=sensor_names)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

### building lstm model and train

In [10]:
class IoTPredictiveMaintenanceLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :]) 
        return out

model = IoTPredictiveMaintenanceLSTM(input_size=21, hidden_size=32, num_layers=2)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [13]:
epochs = 60
batch_size = 64
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

print("മോഡൽ കൃത്യതയോടെ ട്രെയിൻ ചെയ്യുന്നു...")
for epoch in range(epochs):
    model.train()
    # ഡാറ്റ ഷഫിൾ ചെയ്ത് ട്രെയിൻ ചെയ്യുന്നു
    permutation = torch.randperm(X_train_t.size()[0])
    
    for i in range(0, X_train_t.size()[0], batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X_train_t[indices], y_train_t[indices]
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

# പുതിയ പെർഫെക്റ്റ് മോഡൽ സേവ് ചെയ്യുന്നു
torch.save(model.state_dict(), 'lstm_model.pth')
print("🎉 വിജയകരമായി ട്രെയിൻ ചെയ്ത് സേവ് ചെയ്തു! ഇനി ബാക്കെൻഡ് റീസ്റ്റാർട്ട് ചെയ്യാം.")



മോഡൽ കൃത്യതയോടെ ട്രെയിൻ ചെയ്യുന്നു...
🎉 വിജയകരമായി ട്രെയിൻ ചെയ്ത് സേവ് ചെയ്തു! ഇനി ബാക്കെൻഡ് റീസ്റ്റാർട്ട് ചെയ്യാം.
